In [ ]:
# BLOQUE 1 — Parámetros del snapshot GTFS a cargar en Bronze

import re

# Localizo los módulos en el repositorio completo, igual que el notebook Silver.
from pathlib import Path
import sys

dbutils.widgets.text("repo_root", "", "Raíz del repositorio en Workspace")
root_parameter = dbutils.widgets.get("repo_root").strip()
if root_parameter and not Path(root_parameter).is_absolute():
    raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
project_root = next((p for p in roots if (p / "databricks/gtfs_bronze/runtime.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Sincroniza el repositorio completo y configura repo_root; falta gtfs_bronze.")
module_path = str(project_root / "databricks")
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Estos defaults corresponden a mi entorno actual; el Job puede reemplazarlos.
for key, default in {"catalog": "workspace", "schema": "gtfs_bogota"}.items():
    dbutils.widgets.text(key, default)
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
from gtfs_bronze.contracts import quoted_table
quoted_table(f"{catalog}.{schema}.configuration_check")
spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("staging_volume", "staging_gtfs")
staging_volume = dbutils.widgets.get("staging_volume").strip()
quoted_table(f"{catalog}.{schema}.{staging_volume}")
staging_volume_path = f"/Volumes/{catalog}/{schema}/{staging_volume}"

# Las entidades operacionales reciben el mismo snapshot que las referencias.
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text(
    "snapshot_sha256",
    "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952",
    "SHA-256 del snapshot",
)
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()

if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", feed_version):
    raise ValueError("feed_version debe tener el formato AAAA-MM-DD.")
if not re.fullmatch(r"[0-9a-f]{64}", snapshot_sha256):
    raise ValueError("snapshot_sha256 debe ser un hash SHA-256 de 64 caracteres.")

staging_snapshot_path = (
    f"{staging_volume_path}/gtfs_static/"
    f"feed_version={feed_version}/"
    f"sha256={snapshot_sha256}"
)

print(f"Snapshot de Staging: {staging_snapshot_path}")
display(dbutils.fs.ls(staging_snapshot_path))
dbutils.widgets.text("show_diagnostics", "false")
show_diagnostics_parameter = dbutils.widgets.get("show_diagnostics").strip().lower()
if show_diagnostics_parameter not in {"true", "false"}:
    raise ValueError("show_diagnostics debe ser true o false.")
show_diagnostics = show_diagnostics_parameter == "true"


In [0]:
# BLOQUE 2 — Contrato de entidades GTFS operacionales

# Estas entidades tienen mayor volumen y conectan viajes, paradas y geometrías.
operational_entities = [
    {
        "file_name": "shapes.txt",
        "table_name": "bronze_gtfs_shapes",
        "required_columns": ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
    },
    {
        "file_name": "stop_times.txt",
        "table_name": "bronze_gtfs_stop_times",
        "required_columns": ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    },
]

In [ ]:
# BLOQUE 3 — Funciones compartidas de validación y carga Bronze

from gtfs_bronze.runtime import read_gtfs_header, source_frame, load_frame
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, LongType, StringType, StructField, StructType


def load_entity_to_bronze(entity: dict) -> tuple:
    """Conserva la fuente como texto y rechaza cargas parciales sin duplicar filas."""
    source_path = f"{staging_snapshot_path}/{entity['file_name']}"
    table_name = f"{catalog}.{schema}.{entity['table_name']}"
    source_columns = read_gtfs_header(source_path)
    missing_columns = sorted(set(entity['required_columns']) - set(source_columns))
    if missing_columns:
        raise RuntimeError(f"Contrato GTFS incumplido en {entity['file_name']}. Faltan columnas: {', '.join(missing_columns)}")
    count, reused = load_frame(
        spark, source_frame(spark, source_path, source_columns), table=table_name,
        source_path=source_path, feed_version=feed_version, sha256=snapshot_sha256,
    )
    return entity['file_name'], table_name, count, reused


load_result_schema = StructType([
    StructField("source_file_name", StringType(), False),
    StructField("bronze_table_name", StringType(), False),
    StructField("source_row_count", LongType(), False),
    StructField("snapshot_already_loaded", BooleanType(), False),
])


In [0]:
# BLOQUE 4 — Carga idempotente de shapes.txt

shapes_result = load_entity_to_bronze(operational_entities[0])

display(spark.createDataFrame([shapes_result], schema=load_result_schema))

In [0]:
# BLOQUE 5 — Carga idempotente de stop_times.txt

# stop_times conecta los viajes con las paradas y es la entidad más relevante para Silver.
stop_times_result = load_entity_to_bronze(operational_entities[1])

display(spark.createDataFrame([stop_times_result], schema=load_result_schema))

In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # BLOQUE 6 — Verificación de las entidades operacionales persistidas

    bronze_stop_times_table = f"{catalog}.{schema}.bronze_gtfs_stop_times"

    display(
        spark.table(bronze_stop_times_table)
        .filter(F.col("source_sha256") == snapshot_sha256)
        .orderBy("trip_id", "stop_sequence")
        .limit(20)
    )

    display(spark.sql(f"describe detail {bronze_stop_times_table}"))


In [ ]:
# Diagnóstico opcional; notebook 07 mantiene la compuerta obligatoria.
if show_diagnostics:
    # BLOQUE 7 — Validación de persistencia de las entidades operacionales

    from pyspark.sql import functions as F
    from pyspark.sql.types import LongType, StringType, StructField, StructType

    operational_tables = [
        f"{catalog}.{schema}.bronze_gtfs_shapes",
        f"{catalog}.{schema}.bronze_gtfs_stop_times",
    ]

    validation_rows = []

    for table_name in operational_tables:
        snapshot_df = (
            spark.table(table_name)
            .filter(F.col("source_sha256") == snapshot_sha256)
        )

        validation_rows.append((
            table_name,
            snapshot_df.count(),
            snapshot_df.select("source_feed_version").first()["source_feed_version"],
            snapshot_df.select("source_file_path").first()["source_file_path"],
        ))

    validation_schema = StructType([
        StructField("bronze_table", StringType(), False),
        StructField("rows_for_snapshot", LongType(), False),
        StructField("source_feed_version", StringType(), False),
        StructField("source_file_path", StringType(), False),
    ])

    validation_df = spark.createDataFrame(
        validation_rows,
        schema=validation_schema,
    )

    display(validation_df)
